<a href="https://colab.research.google.com/github/iortcacha/dotfiles/blob/master/A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [11]:
!pip install pandasql


  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26773 sha256=0be731486a3c43a9f4b8a951fad770f4ff390bbc5106ebf270fa2f84bf818cbe
  Stored in directory: /root/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql


In [12]:
from pandasql import sqldf
q = lambda query: sqldf(query, globals()) # Shortcut to run SQL

Load and check data

In [6]:
transactions = pd.read_csv("transactions.csv")
users = pd.read_csv("users.csv")
currency_details = pd.read_csv("currency_details.csv")
fx_rates = pd.read_csv("fx_rates.csv")

print(transactions.shape)
print(users.shape)
print(currency_details.shape)
print(fx_rates.shape

(688651, 12)
(10300, 11)
(208, 4)
(84, 3)


Get the first transaction per client

In [7]:
query_first_tx = """
WITH ranked AS (
    SELECT
        t.*,
        ROW_NUMBER() OVER (
            PARTITION BY t.user_id
            ORDER BY t.created_date ASC
        ) AS rn
    FROM transactions t
)
SELECT *
FROM ranked
WHERE rn = 1
"""

Validating that the number of first transactions is the same as the number of unique id



In [13]:
first_tx = q(query_first_tx)
first_tx.shape


(8021, 13)

In [14]:
q("""
SELECT COUNT(DISTINCT user_id) AS total_users
FROM transactions
""")

,total_users
0,8021


Converting the amount to USD. Create new column for real amount, converting the amount into integer numbers, using its exponent. Followed by converting all currencys to USD

In [15]:
query_convert = """
SELECT
    f.user_id,
    f.type,
    f.state,
    f.currency,
    f.amount,
    cd.exponent,
    fr.rate,

    f.amount / pow(10, COALESCE(cd.exponent, 2)) AS amount_real,

    CASE
        WHEN f.currency = 'USD' THEN
            f.amount / pow(10, COALESCE(cd.exponent, 2))
        ELSE
            (f.amount / pow(10, COALESCE(cd.exponent, 2))) / fr.rate

    END AS amount_usd

FROM first_tx f
LEFT JOIN currency_details cd
    ON f.currency = cd.currency
LEFT JOIN fx_rates fr
    ON f.currency = fr.ccy
   AND fr.base_ccy = 'USD'
"""

In [16]:
converted = q(query_convert)
converted.head()

,USER_ID,TYPE,STATE,CURRENCY,AMOUNT,exponent,rate,amount_real,amount_usd
0,000e88bb-d302-4fdc-b757-2b1a2c33e7d6,TOPUP,REVERTED,DKK,100,2.0,0.164050,1.0,6.095713
1,001032e0-8071-4baf-95b9-e50214665c2e,TOPUP,REVERTED,EUR,100,2.0,1.177809,1.0,0.849034
2,00131af8-66f0-4526-8b5f-dc2fdb26c7d7,TOPUP,REVERTED,EUR,100,2.0,1.177809,1.0,0.849034
3,001926be-3245-43fa-86dd-b40ee160b6f9,TOPUP,REVERTED,GBP,100,2.0,1.319906,1.0,0.757630
4,001cc034-5730-47c6-a70c-25f42249c9ee,TOPUP,REVERTED,RON,100,2.0,0.259904,1.0,3.847572


We will select card payments that are completed and higher than 10 USD to calculate the kpi

In [18]:
query_kpi = """
SELECT
    COUNT(DISTINCT CASE
        WHEN c.type = 'CARD_PAYMENT'
         AND c.state = 'COMPLETED'
         AND c.amount_usd > 10
        THEN c.user_id
    END) * 100.0 / COUNT(DISTINCT u.id) AS transaction_success_rate_kpi
FROM users u
LEFT JOIN converted c
    ON u.id = c.user_id
"""

In [19]:
q(query_kpi)

,transaction_success_rate_kpi
0,0.194175


In [20]:
# Low KPI, lets check how many of the first transactions are card payments
q("""
SELECT
    TYPE,
    COUNT(*) AS num_users,
    COUNT(*) * 100.0 / (SELECT COUNT(*) FROM first_tx) AS pct
FROM first_tx
GROUP BY TYPE
ORDER BY num_users DESC
""")

,TYPE,num_users,pct
0,TOPUP,7582,94.526867
1,CARD_PAYMENT,264,3.291360
2,P2P,113,1.408802
3,BANK_TRANSFER,39,0.486224
4,ATM,23,0.286747
